In [7]:
trees = [4799, 6061, 4384, 1644, 1553, 11764, 10868, 2519, 7943, 2020, 11581, 12445, 4972, 8567, 14090, 4161, 13501, 2965, 4113, 13488, 6130, 6697, 9423, 5836, 9384, 14117, 5216, 1308, 13095, 1250, 10993, 13886, 9799, 10151, 5106, 6304, 14207, 106, 2897, 10994, 3731, 3854, 3208, 9292, 1966, 11723, 3241, 13361, 1080, 3861, 13624, 1568, 8146, 12715, 1845, 10579, 8637, 8952, 13599, 9244, 12985, 10273, 1089, 984, 8300, 2763, 8878, 10821, 375, 6177, 3629, 12528, 8927, 2379, 5190, 4864, 287, 6435, 5478, 11751, 12582, 10667, 8936, 5074, 7721, 3585, 5469, 11486, 9641, 2817, 5944, 13959, 9646, 3879, 11169, 4145, 4524, 23, 12813, 10334, 2590, 4220, 7335, 11749, 9699, 7398, 3636, 6480, 13711, 6278, 4014, 4475, 7025, 5467, 5815, 9400, 7332, 770, 11488, 13285, 739, 13841, 1246, 13724, 8427, 4832, 4564, 12579, 11293, 12091, 827, 3430, 13510, 3893, 5429, 10257, 7552, 7133, 4331, 13446, 13027, 12667, 2481, 503, 8820, 2169, 5503, 2508, 1158, 737, 10149, 6243, 8110, 5737, 13213, 10470, 3494, 6549, 6367, 6706, 6448, 13540, 12258, 11849, 7970, 13005, 14246, 10200, 1495, 10794, 5402, 402, 11177, 2523, 4368, 1391, 5011, 4680, 1679, 470, 9624, 3469, 5554, 13411, 1605, 3943, 12637, 1174, 7049, 2717, 13244, 1081, 1347, 2252, 13532, 4656, 11498, 9707, 10668, 5107]

In [6]:
import os
import zipfile
from math import ceil
import utils.constants as constants

In [10]:
inter_path = lambda tree_id:  f"/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/{tree_id}.pkl"
final_tree = lambda model, tree_id: f"/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/{model.replace('/', '-')}/complete/{tree_id}_checked.pkl"
zip_dir = "/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/"

models = constants.MODELS
if len(models) != 3:
    raise RuntimeError(f"Expected exactly 3 models, but found {len(models)}")

os.makedirs(zip_dir, exist_ok=True)

# --- Split `trees` into 3 roughly equal parts ---
n_trees = len(trees)
part_size = ceil(n_trees / 3)
parts = [
    trees[i * part_size : (i + 1) * part_size]
    for i in range(3)
]

for part_idx, segment in enumerate(parts, start=1):
    # 1) Zip intermediate trees for this part
    inter_zip_name = os.path.join(zip_dir, f"part{part_idx}_trees.zip")
    with zipfile.ZipFile(inter_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for tree_id in segment:
            src = inter_path(tree_id)
            if os.path.exists(src):
                zf.write(src, arcname=os.path.basename(src))
            else:
                print(f"Warning: intermediate file not found → {src}")
    print(f"Created {inter_zip_name} (contains {len(segment)} intermediate trees)")

    # 2) For each model, zip its “checked” trees for this part
    for model in models:
        model_zip_name = os.path.join(
            zip_dir,
            f"part{part_idx}_{model.replace('/', '-')}.zip"
        )
        with zipfile.ZipFile(model_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for tree_id in segment:
                src = final_tree(model, tree_id)
                if os.path.exists(src):
                    zf.write(src, arcname=os.path.basename(src))
                else:
                    print(f"Warning: final file not found → {src}")
        print(f"Created {model_zip_name} (contains {len(segment)} trees for model '{model}')")

# --- Summary ---
print("\nSummary of partitions:")
for idx, segment in enumerate(parts, start=1):
    print(f"  Part {idx}: {len(segment)} tree IDs → {segment}")

Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_trees.zip (contains 67 intermediate trees)
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_google-gemma-3-1b-it.zip (contains 67 trees for model 'google/gemma-3-1b-it')
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_google-gemma-3-12b-it.zip (contains 67 trees for model 'google/gemma-3-12b-it')
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_mistralai-Mistral-7B-Instruct-v0.2.zip (contains 67 trees for model 'mistralai/Mistral-7B-Instruct-v0.2')
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part2_trees.zip (contains 67 intermediate trees)
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part2_google-gemma-3-1b-it.zip (contains 67 trees for model 'google/gemma-3-1b-it')
Created /vol/bitbucket/lst20/POPQA_treenodes

In [12]:
tree_3 = [5429, 10257, 7552, 7133, 4331, 13446, 13027, 12667, 2481, 503, 8820, 2169, 5503, 2508, 1158, 737, 10149, 6243, 8110, 5737, 13213, 10470, 3494, 6549, 6367, 6706, 6448, 13540, 12258, 11849, 7970, 13005, 14246, 10200, 1495, 10794, 5402, 402, 11177, 2523, 4368, 1391, 5011, 4680, 1679, 470, 9624, 3469, 5554, 13411, 1605, 3943, 12637, 1174, 7049, 2717, 13244, 1081, 1347, 2252, 13532, 4656, 11498, 9707, 10668, 5107]
inter_path = lambda tree_id:  f"/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/{tree_id}.pkl"
final_tree = lambda model, tree_id: f"/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/{model.replace('/', '-')}/complete/{tree_id}_checked.pkl"
zip_dir = "/vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/"

models = constants.MODELS
if len(models) != 3:
    raise RuntimeError(f"Expected exactly 3 models, but found {len(models)}")

os.makedirs(zip_dir, exist_ok=True)

# --- Split `trees` into 3 roughly equal parts ---
n_trees = len(tree_3)
part_size = ceil(n_trees / 2)
parts = [
    tree_3[i * part_size : (i + 1) * part_size]
    for i in range(2)
]

for part_idx, segment in enumerate(parts, start=1):
    # 1) Zip intermediate trees for this part
    inter_zip_name = os.path.join(zip_dir, f"part_3-{part_idx}_trees.zip")
    with zipfile.ZipFile(inter_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for tree_id in segment:
            src = inter_path(tree_id)
            if os.path.exists(src):
                zf.write(src, arcname=os.path.basename(src))
            else:
                print(f"Warning: intermediate file not found → {src}")
    print(f"Created {inter_zip_name} (contains {len(segment)} intermediate trees)")

    # 2) For each model, zip its “checked” trees for this part
    for model in models:
        model_zip_name = os.path.join(
            zip_dir,
            f"part_3-{part_idx}_{model.replace('/', '-')}.zip"
        )
        with zipfile.ZipFile(model_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for tree_id in segment:
                src = final_tree(model, tree_id)
                if os.path.exists(src):
                    zf.write(src, arcname=os.path.basename(src))
                else:
                    print(f"Warning: final file not found → {src}")
        print(f"Created {model_zip_name} (contains {len(segment)} trees for model '{model}')")

# --- Summary ---
print("\nSummary of partitions:")
for idx, segment in enumerate(parts, start=1):
    print(f"  Part {idx}: {len(segment)} tree IDs → {segment}")

Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part_3-1_trees.zip (contains 33 intermediate trees)
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part_3-1_google-gemma-3-1b-it.zip (contains 33 trees for model 'google/gemma-3-1b-it')
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part_3-1_google-gemma-3-12b-it.zip (contains 33 trees for model 'google/gemma-3-12b-it')
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part_3-1_mistralai-Mistral-7B-Instruct-v0.2.zip (contains 33 trees for model 'mistralai/Mistral-7B-Instruct-v0.2')
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part_3-2_trees.zip (contains 33 intermediate trees)
Created /vol/bitbucket/lst20/POPQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part_3-2_google-gemma-3-1b-it.zip (contains 33 trees for model 'google/gemma-3-1b-it')
Created /vol/bitbucket/lst

In [17]:
for i, t in enumerate(trees):
    if t == 5429 or t == "5429":
        print(t, i)
        break
    

5429 134


# TQA paraprefix+dialect

In [ ]:
tqa_trees = [11004, 27830, 73939, 39947, 66202, 21275, 61380, 73290, 70043, 26740, 36212, 19349, 13490, 51680, 27219, 8041, 57646, 33629, 66277, 59347, 34244, 54567, 1362, 70450, 34659, 48758, 62464, 26270, 24253, 65047, 6665, 41606, 50603, 22746, 62480, 2862, 32609, 40504, 67196, 19637, 47205, 57832, 40090, 9648, 67802, 4987, 30155, 61296, 3695, 4589, 26934, 22039, 14197, 213, 38679, 42583, 32434, 39972, 39276, 28164, 68612, 56971, 3198, 67016, 51481, 22734, 7503, 71075, 1784, 27234, 49644, 46386, 27937, 25968, 52760, 61520, 14682, 67994, 58875, 6440, 37256, 61583, 53183, 20464, 5010, 54122, 8955, 74082, 9706, 26522, 75, 8838, 20790, 5754, 21088, 15673, 21753, 21025, 70565, 40478, 8081, 55215, 67970, 56804, 96, 293, 22690, 707, 71111, 23621, 69548, 51718, 54097, 15722, 13796, 597, 69941, 4042, 13321, 2103, 63867, 817, 53321, 22710, 36889, 1638, 73308, 5843, 62043, 28272, 68041, 24835, 49592, 40454, 43933, 12777, 16357, 5413, 65745, 73165, 8868, 21246, 8629, 62732, 44943, 12531, 54802, 33358, 24964, 6663, 29563, 54135, 56660, 64177, 16979, 39169, 722, 29898, 17103, 17562, 34167, 18862, 58293, 17214, 48875, 10892, 10081, 36588, 41434, 18155, 39810, 33538, 64633, 32220, 2352, 36723, 45026, 28235, 52737, 40627, 34548, 10768, 56310, 68812, 44033, 68544, 30858, 28412, 59023, 32634, 18169, 2203, 64409, 43112, 41302, 49323, 24646, 6946, 45963, 13911, 57588, 3025, 15749, 31396, 27728, 32549, 63009, 27324, 66440, 9664, 34474, 65693, 61172, 72535, 63151, 53900, 54273, 67033, 48030, 67452, 4488, 53030, 50936, 3794, 40760, 26750, 17624, 8755, 547, 39696, 2891, 20202, 9881, 41819, 14743, 70786, 18393, 60888, 52844, 11371, 68639, 2435, 23310, 61217, 57618, 31467, 58195, 16676, 28567, 50387, 3291, 41908, 66872, 73335, 64329, 8140, 49146, 12300, 59299, 29972, 55716, 5374, 49927, 15986, 37644, 11026, 37926, 45070, 60548, 26551, 25782, 65438, 43647, 43922, 5064, 67770, 18801, 37337, 6064, 61789, 52336, 55603, 69181, 53099, 14167, 66443, 62405, 32536, 39928, 34132, 48794, 43399, 37607, 13668, 9505, 44673, 28924, 30782, 13387, 52891, 31741, 57248, 34631, 26391, 35410, 43839, 64090, 30529, 25432, 19689, 43631, 41977, 29666, 63326, 49255, 27940, 48531, 29820, 17485, 58767, 3369, 16029, 26878, 46799, 66878, 38038, 59948, 25898, 2555, 34210, 64739, 54534, 30768, 24452, 52068, 23038, 51059, 7749, 18879, 51912, 45520, 63434, 64398, 40031, 62970, 52686, 12236, 38841, 62421, 56886, 58461, 43629, 23919, 37680, 2232, 66150, 22979, 41390, 3283, 4638, 7675, 56486, 69211, 65276, 5120, 66415, 59981, 15317, 54764, 4135, 34163, 47571, 42962, 61906, 52384, 62461, 31849, 55667, 39007, 49972, 42393, 68369, 72017, 14020, 7473, 48726, 50193, 56368, 55738, 13721, 12489, 25411, 42758, 8095, 36437, 61277, 51654, 42104, 11869, 47626, 12741, 46717, 38950, 55426, 35764, 62994, 73153, 73940, 43792, 43454, 3212, 25816, 41638, 16533, 34776, 71946, 3313, 485, 25104, 35320, 37475, 24795, 24032, 27810, 50618, 22838, 47056, 63617, 48655, 44282, 20186, 40215, 10326, 3775, 19126, 4150, 61013, 73214, 27183, 19367, 30638, 40267, 29792, 73178, 32821, 31775, 52143, 12711, 36469, 57794, 28446, 9363, 58637, 3749, 5102, 7591, 46680, 25649, 23650, 72466, 9883, 51928, 40699, 46775, 62552, 22639, 30348, 28581, 5105, 23464, 26107, 18912, 68649, 21435, 29246, 15967, 17621, 57867, 61788, 53359, 545, 29939, 14054, 58739, 73245, 13755, 8509, 21093, 27751, 225, 582, 44488, 51629, 48830, 64692, 36912, 55420, 24528, 34992, 37887, 14179, 29879, 7989, 21037, 10192, 59170, 67261, 66047, 63719, 2579, 38646, 10588, 69306, 49501, 35717, 43614, 20754, 32334, 12533, 15538, 22571, 10499, 17106, 4223, 16823, 51426, 845, 62389, 59400, 73112, 20471, 31302, 63105, 33664, 9146, 60199, 31192, 67069, 11453, 37126, 69514, 47091, 33459, 64913, 15008, 38498, 13926, 47615, 58543, 4768, 51826, 3474, 35849, 22503, 48926]
first_200_tqa = tqa_trees[:200]
inter_path = lambda tree_id:  f"/vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/tree/{tree_id}.pkl"
final_tree = lambda model, tree_id: f"/vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/{model.replace('/', '-')}/complete/{tree_id}_checked.pkl"
zip_dir = "/vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/"

models = constants.MODELS
if len(models) != 3:
    raise RuntimeError(f"Expected exactly 3 models, but found {len(models)}")

os.makedirs(zip_dir, exist_ok=True)

# --- Split `trees` into 3 roughly equal parts ---
n_trees = len(first_200_tqa)
part_size = ceil(n_trees / 3)
parts = [
    first_200_tqa[i * part_size : (i + 1) * part_size]
    for i in range(3)
]

for part_idx, segment in enumerate(parts, start=1):
    # 1) Zip intermediate trees for this part
    inter_zip_name = os.path.join(zip_dir, f"part{part_idx}_trees.zip")
    with zipfile.ZipFile(inter_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for tree_id in segment:
            src = inter_path(tree_id)
            if os.path.exists(src):
                zf.write(src, arcname=os.path.basename(src))
            else:
                print(f"Warning: intermediate file not found → {src}")
    print(f"Created {inter_zip_name} (contains {len(segment)} intermediate trees)")

    # 2) For each model, zip its “checked” trees for this part
    for model in models:
        model_zip_name = os.path.join(
            zip_dir,
            f"part{part_idx}_{model.replace('/', '-')}.zip"
        )
        with zipfile.ZipFile(model_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for tree_id in segment:
                src = final_tree(model, tree_id)
                if os.path.exists(src):
                    zf.write(src, arcname=os.path.basename(src))
                else:
                    print(f"Warning: final file not found → {src}")
        print(f"Created {model_zip_name} (contains {len(segment)} trees for model '{model}')")

# --- Summary ---
print("\nSummary of partitions:")
for idx, segment in enumerate(parts, start=1):
    print(f"  Part {idx}: {len(segment)} tree IDs → {segment}")

Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_trees.zip (contains 67 intermediate trees)
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_google-gemma-3-1b-it.zip (contains 67 trees for model 'google/gemma-3-1b-it')
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_google-gemma-3-12b-it.zip (contains 67 trees for model 'google/gemma-3-12b-it')
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part1_mistralai-Mistral-7B-Instruct-v0.2.zip (contains 67 trees for model 'mistralai/Mistral-7B-Instruct-v0.2')
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part2_trees.zip (contains 67 intermediate trees)
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/part2_google-gemma-3-1b-it.zip (contains 67 trees for model 'google/gemma-3-1b-it')
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/g

In [19]:
import os
import zipfile
from math import ceil

# ──────────────────────────────────────────────────────────────────────────────
# Assume these two lambdas and `models` are defined exactly as before:
first_200_tqa = tqa_trees[:200]

inter_path = lambda tree_id: (
    f"/vol/bitbucket/lst20/TQA_treenodes/"
    f"para-prefix/gemma3-12b_perturb/3_2_0/tree/{tree_id}.pkl"
)
final_tree = lambda model, tree_id: (
    f"/vol/bitbucket/lst20/TQA_treenodes/"
    f"para-prefix/gemma3-12b_perturb/3_2_0/"
    f"{model.replace('/', '-')}/complete/{tree_id}_checked.pkl"
)

models = constants.MODELS  # e.g. ["modelA", "modelB", "modelC"]
if len(models) != 3:
    raise RuntimeError(f"Expected exactly 3 models, but found {len(models)}")

zip_dir = "/vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/"
os.makedirs(zip_dir, exist_ok=True)
# ──────────────────────────────────────────────────────────────────────────────

# 1) Create a single ZIP for **all** intermediate trees (first 200)
inter_zip_name = os.path.join(zip_dir, "trees.zip")
with zipfile.ZipFile(inter_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    missing_inter = 0
    for tree_id in first_200_tqa:
        src = inter_path(tree_id)
        if os.path.exists(src):
            # arcname = just the basename (e.g. "12345.pkl") so that inside the zip
            # you don’t get the full directory hierarchy.
            zf.write(src, arcname=os.path.basename(src))
        else:
            print(f"Warning: intermediate file not found → {src}")
            missing_inter += 1

print(
    f"Created {inter_zip_name}  "
    f"(requested {len(first_200_tqa)}, missing {missing_inter})"
)

# 2) For each model, create a single ZIP containing all 200 “checked” trees
for model in models:
    safe_model_name = model.replace("/", "-")
    model_zip_name = os.path.join(zip_dir, f"{safe_model_name}.zip")

    with zipfile.ZipFile(model_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        missing_final = 0
        for tree_id in first_200_tqa:
            src = final_tree(model, tree_id)
            if os.path.exists(src):
                zf.write(src, arcname=os.path.basename(src))
            else:
                print(f"Warning: final file not found → {src}")
                missing_final += 1

    print(
        f"Created {model_zip_name}  "
        f"(requested {len(first_200_tqa)}, missing {missing_final})"
    )

# ──────────────────────────────────────────────────────────────────────────────
# At this point, you have:
#   /vol/…/3_2_0/trees.zip             ← all 200 intermediate .pkl files
#   /vol/…/3_2_0/modelA.zip           ← all 200 modelA _checked.pkl
#   /vol/…/3_2_0/modelB.zip           ← all 200 modelB _checked.pkl
#   /vol/…/3_2_0/modelC.zip           ← all 200 modelC _checked.pkl
# ──────────────────────────────────────────────────────────────────────────────


Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/trees.zip  (requested 200, missing 0)
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/google-gemma-3-1b-it.zip  (requested 200, missing 0)
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/google-gemma-3-12b-it.zip  (requested 200, missing 0)
Created /vol/bitbucket/lst20/TQA_treenodes/para-prefix/gemma3-12b_perturb/3_2_0/mistralai-Mistral-7B-Instruct-v0.2.zip  (requested 200, missing 64)
